In [1]:
from pathlib import Path
import sys

project_dir = Path.cwd()

required_files = [
    "README.md",
    "data/evaluation/development_v1.json",
]

if not all((project_dir / name).is_file() for name in required_files):
    raise RuntimeError("Open this notebook from the project root.")

print("Project folder:", project_dir)
print("Python executable:", sys.executable)
print("Access-control lab ready.")

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Access-control lab ready.


In [2]:
import json

restricted_dir = project_dir / "data" / "access_control" / "restricted"
restricted_dir.mkdir(parents=True, exist_ok=True)

restricted_path = restricted_dir / "hr_review_policy.txt"

restricted_text = """\
SYNTHETIC TRAINING DOCUMENT
Company: Example Workplace
Document ID: HR-RESTRICTED-001
Title: Internal HR Review Procedure

The internal HR review procedure uses the reference code HR-REVIEW-742.
HR administrators must record review decisions in the restricted HR register.
"""

access_policy = {
    "schema_version": "1.0",
    "identity_mode": "simulated role input; no authentication",
    "roles": ["employee", "hr_admin"],
    "documents": [
        {
            "id": "HR-001",
            "path": "data/synthetic/leave_policy.txt",
            "allowed_roles": ["employee", "hr_admin"],
        },
        {
            "id": "HR-002",
            "path": "data/synthetic/remote_work_policy.txt",
            "allowed_roles": ["employee", "hr_admin"],
        },
        {
            "id": "IT-001",
            "path": "data/synthetic/it_support_policy.txt",
            "allowed_roles": ["employee", "hr_admin"],
        },
        {
            "id": "HR-RESTRICTED-001",
            "path": "data/access_control/restricted/hr_review_policy.txt",
            "allowed_roles": ["hr_admin"],
        },
    ],
}

policy_path = project_dir / "data" / "access_control" / "policy_v1.json"


def write_if_matching_or_absent(path, content):
    if path.exists():
        if path.read_text(encoding="utf-8") != content:
            raise RuntimeError(f"Existing file differs: {path.name}")
        print("Already matches:", path.name)
    else:
        path.write_text(content, encoding="utf-8")
        print("Created:", path.name)


write_if_matching_or_absent(restricted_path, restricted_text)

write_if_matching_or_absent(
    policy_path,
    json.dumps(access_policy, indent=2, ensure_ascii=False) + "\n",
)

print("Policy documents:", len(access_policy["documents"]))
print("Configured roles:", ", ".join(access_policy["roles"]))

Already matches: hr_review_policy.txt
Already matches: policy_v1.json
Policy documents: 4
Configured roles: employee, hr_admin


In [3]:
class AccessDeniedError(PermissionError):
    """The supplied role is not recognized."""


class AccessPolicyError(ValueError):
    """The access policy is invalid."""


def load_authorized_passages(role):
    policy = json.loads(policy_path.read_text(encoding="utf-8"))

    # Validate the policy before reading any document contents.
    roles = policy.get("roles")
    documents = policy.get("documents")

    if (
        not isinstance(roles, list)
        or not roles
        or any(not isinstance(item, str) or not item for item in roles)
        or len(set(roles)) != len(roles)
        or not isinstance(documents, list)
        or not documents
    ):
        raise AccessPolicyError("Invalid role or document definitions.")

    if not isinstance(role, str) or role not in roles:
        raise AccessDeniedError("Unrecognized role.")

    validated_documents = []
    document_ids = set()
    document_paths = set()
    root = project_dir.resolve()

    for document in documents:
        if not isinstance(document, dict):
            raise AccessPolicyError("Invalid document entry.")

        document_id = document.get("id")
        relative_path = document.get("path")
        allowed_roles = document.get("allowed_roles")

        if (
            not isinstance(document_id, str)
            or not document_id
            or not isinstance(relative_path, str)
            or not relative_path
            or not isinstance(allowed_roles, list)
            or not allowed_roles
            or any(
                not isinstance(item, str) or item not in roles
                for item in allowed_roles
            )
        ):
            raise AccessPolicyError("Invalid document permissions.")

        path = Path(relative_path)

        if path.is_absolute() or path.drive or path.root:
            raise AccessPolicyError("Document paths must be relative.")

        resolved_path = (root / path).resolve()

        if not resolved_path.is_relative_to(root):
            raise AccessPolicyError("Document path escapes project root.")

        if document_id in document_ids or resolved_path in document_paths:
            raise AccessPolicyError("Duplicate document ID or path.")

        document_ids.add(document_id)
        document_paths.add(resolved_path)
        validated_documents.append(
            (document_id, resolved_path, allowed_roles)
        )

    loaded = []
    read_document_ids = []

    for document_id, path, allowed_roles in validated_documents:
        if role not in allowed_roles:
            continue  # Deny before reading this document's contents.

        text = path.read_text(encoding="utf-8")
        read_document_ids.append(document_id)

        _, separator, body = text.partition("\n\n")

        if not separator or not body.strip():
            raise ValueError(f"Missing policy body: {document_id}")

        for line_number, line in enumerate(body.splitlines(), start=1):
            sentence = line.strip()

            if sentence:
                loaded.append({
                    "document_id": document_id,
                    "passage_id": f"{document_id}:{line_number}",
                    "text": sentence,
                })

    return loaded, read_document_ids


for role in ["employee", "hr_admin"]:
    authorized_passages, read_ids = load_authorized_passages(role)

    print(f"\nRole: {role}")
    print("Documents read:", ", ".join(read_ids))
    print("Passages loaded:", len(authorized_passages))


Role: employee
Documents read: HR-001, HR-002, IT-001
Passages loaded: 9

Role: hr_admin
Documents read: HR-001, HR-002, IT-001, HR-RESTRICTED-001
Passages loaded: 11


In [4]:
import re

STOP_WORDS = {
    "a", "an", "the", "is", "are", "do", "does", "how",
    "what", "which", "of", "to", "for", "per", "in",
    "on", "and", "or", "with",
}

def access_tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP_WORDS


def retrieve_authorized(question, role, top_k=2):
    if type(top_k) is not int or not 1 <= top_k <= 10:
        raise ValueError("top_k must be an integer from 1 through 10.")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("A non-empty query is required.")

    query_tokens = access_tokens(question)

    if not query_tokens:
        raise ValueError("Query has no searchable tokens.")

    # Role filtering happens before document contents are read.
    eligible_passages, read_ids = load_authorized_passages(role)
    results = []

    for passage in eligible_passages:
        matched_words = query_tokens & access_tokens(passage["text"])

        if matched_words:
            results.append({
                **passage,
                "score": len(matched_words),
            })

    results.sort(key=lambda item: (-item["score"], item["passage_id"]))

    return {
        "results": results[:top_k],
        "read_document_ids": read_ids,
    }


restricted_question = (
    "What is the reference code for the internal HR review procedure?"
)

employee_response = retrieve_authorized(
    restricted_question, "employee"
)
admin_response = retrieve_authorized(
    restricted_question, "hr_admin"
)

# Check all employee-eligible passages, not only the top results.
employee_passages, employee_read_ids = load_authorized_passages("employee")

assert "HR-RESTRICTED-001" not in employee_read_ids
assert all(
    passage["document_id"] != "HR-RESTRICTED-001"
    and "HR-REVIEW-742" not in passage["text"]
    for passage in employee_passages
)

assert all(
    result["document_id"] != "HR-RESTRICTED-001"
    and "HR-REVIEW-742" not in result["text"]
    for result in employee_response["results"]
)

assert admin_response["results"][0]["passage_id"] == "HR-RESTRICTED-001:1"
assert "HR-REVIEW-742" in admin_response["results"][0]["text"]

# General-policy retrieval remains available to both roles.
for role in ["employee", "hr_admin"]:
    general_response = retrieve_authorized(
        "How many annual leave days do full-time employees receive?",
        role,
    )
    assert general_response["results"][0]["passage_id"] == "HR-001:1"

# Unknown roles must be rejected.
try:
    retrieve_authorized(restricted_question, "unknown_role")
except AccessDeniedError:
    print("PASS: Unknown role denied.")
else:
    raise AssertionError("Unknown role was not denied.")

print("PASS: Employee-eligible passages exclude restricted content.")
print("PASS: Employee retrieval excludes the restricted marker.")
print("PASS: HR administrator retrieves the restricted reference code.")
print("PASS: Both roles can retrieve the general leave policy.")
print("Employee result count:", len(employee_response["results"]))

PASS: Unknown role denied.
PASS: Employee-eligible passages exclude restricted content.
PASS: Employee retrieval excludes the restricted marker.
PASS: HR administrator retrieves the restricted reference code.
PASS: Both roles can retrieve the general leave policy.
Employee result count: 0


In [5]:
from unittest.mock import patch

original_read_text = Path.read_text
restricted_resolved = restricted_path.resolve()
attempted_reads = []


def guarded_read_text(path, *args, **kwargs):
    resolved = path.resolve()
    attempted_reads.append(resolved)

    if resolved == restricted_resolved:
        raise AssertionError(
            "Employee retrieval attempted to read the restricted file."
        )

    return original_read_text(path, *args, **kwargs)


with patch.object(Path, "read_text", guarded_read_text):
    guarded_employee_response = retrieve_authorized(
        restricted_question,
        "employee",
    )

assert restricted_resolved not in attempted_reads

expected_reads = {
    policy_path.resolve(),
    (project_dir / "data/synthetic/leave_policy.txt").resolve(),
    (project_dir / "data/synthetic/remote_work_policy.txt").resolve(),
    (project_dir / "data/synthetic/it_support_policy.txt").resolve(),
}

assert set(attempted_reads) == expected_reads

print("PASS: Employee retrieval read the policy and three allowed documents.")
print("PASS: No restricted-file content read was attempted.")
print("Read attempts:", len(attempted_reads))
print("Employee results:", len(guarded_employee_response["results"]))

PASS: Employee retrieval read the policy and three allowed documents.
PASS: No restricted-file content read was attempted.
Read attempts: 4
Employee results: 0


In [6]:
import hashlib
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc)

access_report = {
    "created_at_utc": timestamp.isoformat(),
    "identity_mode": "simulated role input; no authentication",
    "policy_sha256": hashlib.sha256(
        policy_path.read_bytes()
    ).hexdigest(),
    "restricted_document_sha256": hashlib.sha256(
        restricted_path.read_bytes()
    ).hexdigest(),
    "employee": {
        "read_document_ids": employee_response["read_document_ids"],
        "results": employee_response["results"],
        "intercepted_read_paths": [
            str(path.relative_to(project_dir.resolve()))
            for path in attempted_reads
        ],
        "restricted_read_attempted": (
            restricted_resolved in attempted_reads
        ),
    },
    "hr_admin": {
        "read_document_ids": admin_response["read_document_ids"],
        "results": admin_response["results"],
    },
    "limitations": [
        "Roles are supplied by the caller, not authenticated.",
        "Checks exercise this notebook loader and retrieval path.",
        "Read interception covers Path.read_text calls.",
        "No operating-system permissions or shared caches were tested.",
        "Restricted content is synthetic and intended for public publication.",
    ],
}

evidence_dir = project_dir / "evidence"
evidence_dir.mkdir(exist_ok=True)

access_report_path = evidence_dir / (
    f"access_control_{timestamp.strftime('%Y%m%dT%H%M%S%fZ')}.json"
)

with access_report_path.open("x", encoding="utf-8") as file:
    json.dump(access_report, file, indent=2, ensure_ascii=False)

print("Access evidence saved:", access_report_path)
print(
    "Restricted read attempted:",
    access_report["employee"]["restricted_read_attempted"],
)

Access evidence saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\access_control_20261011T165829690168Z.json
Restricted read attempted: False
